# Evaluate an evolved policy on the TEST brain (cross-brain runs)

Companion to `compare_proofreader_policy.ipynb`, but for a run launched with
`--train-brains … --test-brains …`. It runs the run's final accepted policy on
each **test** brain — a WHOLE brain the reviser never saw — and reports the same
quantification (per-skeleton metric deltas + split-repair coverage), so every
number here is a clean held-out generalization measurement (no within-brain split).

It auto-detects the test brains from the run's `split.json` (`cross_brain` /
`test_brains`) and whether the run was `splits_only` from the ledger, so you only
set `RUN_NAME`. Falls back gracefully to a per-brain run's held-out subset if the
run was NOT cross-brain (with a warning).

## 0. Setup — env, imports, parameters, auto-detect test brains

In [1]:
import os, sys, json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Same env / path setup as compare_proofreader_policy.ipynb.
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../configs/zihan_gcs_token.json"
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"

PROJ = Path("..").resolve()                      # exa-spim-agent/
sys.path.insert(0, str(PROJ))
sys.path.insert(0, str(PROJ.parent / "segmentation-skeleton-metrics" / "src"))
sys.path.insert(0, str(PROJ / "scripts"))

from proofreader_evolve.harness import incremental_scoring as inc
from proofreader_evolve.harness import dataset as ds
from proofreader_evolve.harness import candidate as cand
from proofreader_evolve.harness import scoring
from proofreader_evolve.harness.edit_handler import normalize_edits

# --- Parameters -----------------------------------------------------------
RUN_NAME = "789202_train1_test1_20260628_212929"   # the cross-brain evolution run id
GEN = None                                      # None -> last ACCEPTED generation (best policy)

RUNS = PROJ / "proofreader_evolve" / "runs"
RUN_DIR = RUNS / RUN_NAME
assert RUN_DIR.is_dir(), f"run dir not found: {RUN_DIR}"

# Auto-resolve GEN to the last accepted generation from the ledger.
_ledger = [json.loads(l) for l in (RUN_DIR / "ledger.jsonl").read_text().splitlines() if l.strip()]
if GEN is None:
    _accepted = [r["generation"] for r in _ledger if r.get("accepted")]
    assert _accepted, f"no accepted generation in {RUN_DIR/'ledger.jsonl'}"
    GEN = f"gen{max(_accepted):02d}"
    print(f"GEN auto-resolved to last accepted generation: {GEN} (accepted: {sorted(_accepted)})")

# splits_only mode of the run (drives whether MergeSites are enumerated).
SPLITS_ONLY = bool(_ledger[0].get("splits_only", False))

# Read the split config: cross-brain test brains, mcl, per-brain partition.
_split = json.loads((RUN_DIR / "split.json").read_text())
MCL = int(_split.get("mcl", 100))
CROSS_BRAIN = bool(_split.get("cross_brain", False))
if CROSS_BRAIN:
    TEST_BRAINS = [str(b) for b in (_split.get("test_brains") or [])]
    print(f"CROSS-BRAIN run: test brains = {TEST_BRAINS}; train brains = {_split.get('train_brains')}")
else:
    # Not a cross-brain run: fall back to every brain's held-out SUBSET (per-brain split).
    TEST_BRAINS = list(_split.get("per_brain", {}).keys())
    print(f"[WARN] run is NOT cross-brain (no --test-brains). Falling back to the per-brain "
          f"HELD-OUT subset of: {TEST_BRAINS}. Numbers are the within-brain held-out, not a "
          f"whole-brain test set.")

POLICY = RUN_DIR / GEN / "heuristics.accepted.py"
assert POLICY.exists(), f"accepted policy not found: {POLICY}"
print(f"policy: {POLICY}\nmcl={MCL}  splits_only={SPLITS_ONLY}  generations in ledger={len(_ledger)}")

GEN auto-resolved to last accepted generation: gen39 (accepted: [1, 2, 5, 7, 9, 14, 15, 18, 19, 21, 23, 28, 30, 31, 39])
CROSS-BRAIN run: test brains = ['794491']; train brains = ['789202']
policy: /allen/programs/mindscope/workgroups/auto-model/zihan.zhang/exaspim-agent/exa-spim-agent/proofreader_evolve/runs/789202_train1_test1_20260628_212929/gen39/heuristics.accepted.py
mcl=10  splits_only=True  generations in ledger=40


## 1. Load the policy once (shared across all test brains)

In [2]:
propose_edits, policy_module = cand._load_policy(str(POLICY))
raw_enum = getattr(policy_module, "ENUM_PARAMS", None)
raw_enum = {**raw_enum} if isinstance(raw_enum, dict) else {}
raw_enum.setdefault("max_gap_um", 15.0)
ENUM_PARAMS = ds.resolve_enum_params(raw_enum)
print("resolved ENUM_PARAMS:", ENUM_PARAMS)

resolved ENUM_PARAMS: {'max_gap_um': 15.0, 'split_max_sites': 25000, 'tip_to_shaft': True, 'split_alt_per_pair': 1, 'min_arm_cable_um': 10.0, 'seed_depth_um': 8.0, 'merge_max_sites': 5000, 'max_per_label': 8}


## 2. Per-brain evaluation function

For one brain: load its prepared state + the matching fragment cache, run the
policy (with the SAME image reader the loop used), score baseline-vs-edited on the
held-out neurons, and classify the merge edits against the held-out label→neuron
map. For a cross-brain TEST brain the held-out set is the whole brain, so this is a
pure generalization measurement.

In [3]:
def _pair_key(a, b):
    a, b = str(a), str(b)
    return (a, b) if a <= b else (b, a)

def analyze_brain(brain_id):
    """Run the policy on `brain_id` and return all measurements as a dict."""
    bdir_prepared = RUN_DIR / f"prepared_{brain_id}.pkl"
    assert bdir_prepared.exists(), (
        f"prepared cache missing for test brain {brain_id}: {bdir_prepared} "
        f"(the run must have set this brain up; check it is in split.json per_brain)")
    prepared = inc.PreparedBrain.load(str(bdir_prepared))

    # SAME fragment cache the run enumerated over (validated against brain+mcl).
    cache_path = ds.default_cache_path(brain_id, min_cable_length=MCL)
    fragments_graph, _gt, _payload = ds.load_cached_graphs(
        cache_path, expect_brain=brain_id, expect_mcl=MCL)

    # This brain's neuron partition from split.json. For a cross-brain test brain
    # heldout == all neurons (train == []). For the per-brain fallback it is the
    # within-brain held-out subset.
    pb = _split["per_brain"][brain_id]
    heldout_names = sorted(pb["heldout"])
    train_names = sorted(pb["train"])
    all_gt = sorted(prepared.gt_graphs.keys())
    assert heldout_names, f"brain {brain_id} has no held-out neurons in split.json"

    # IMAGE READER — must match the evolution loop (gen policy gates recall on it).
    try:
        from proofreader_evolve.harness.image_features import LazyImagePatchReader
        from dataset_config import get_img_path
        _prefixes = str(PROJ / "configs" / "exaspim_image_prefixes.json")
        _img_path = get_img_path(brain_id, prefixes_path=_prefixes)
        image_reader = LazyImagePatchReader(_img_path, fragments_graph)
        image_on = True
    except Exception as _e:
        image_reader, image_on = None, False
        print(f"[WARN] {brain_id}: could not build image reader ({_e}); image tiers "
              f"DISABLED -> edits are the geometry-only subset, NOT the true policy.")

    # Enumerate the SAME candidate stream (merges only if the run scored merges).
    # _enumerate_sites_cached returns (split_sites, merge_sites, split_enum_stats);
    # the trailing stats dict is the GT-free enumeration-ceiling info (unused here).
    split_sites, merge_sites, _split_stats = cand._enumerate_sites_cached(
        fragments_graph, ENUM_PARAMS, enumerate_merges=not SPLITS_ONLY)
    sites = list(split_sites) + list(merge_sites)
    ctx = {
        "max_gap_um": ENUM_PARAMS["max_gap_um"], "enum_params": dict(ENUM_PARAMS),
        "fragments_graph": fragments_graph, "n_split_sites": len(split_sites),
        "n_merge_sites": len(merge_sites),
        "node_radius": getattr(fragments_graph, "node_radius", None),
        "read_image_patch": image_reader,
    }
    edits = normalize_edits(propose_edits(sites, ctx))
    if SPLITS_ONLY:
        edits = [e for e in edits if e.get("kind") == "merge_labels"]

    # Score baseline vs edited, RESTRICTED to this brain's held-out neurons.
    base = inc.score_incremental(prepared, label_pairs=None, gt_swc_names=heldout_names, verbose=False)
    edited = inc.score_incremental(prepared, edits=edits, gt_swc_names=heldout_names, verbose=False)

    # Split-repair classification against the HELD-OUT label->neuron map (the gate signal).
    heldout_map = inc.label_gt_counts(prepared, gt_names=heldout_names)
    repair = inc.classify_merge_edits(edits, heldout_map)

    # PER-SITE coverage: distinct reachable real splits fixed vs remaining (held-out).
    accepted = {_pair_key(e.get("label_a"), e.get("label_b"))
                for e in edits if e.get("kind") == "merge_labels" and e.get("label_a")}
    def _dom(lbl):
        c = heldout_map.get(str(lbl)); return max(c, key=c.get) if c else None
    fixed = remaining = 0
    for s in split_sites:
        la, lb = str(s.label_a), str(s.label_b)
        da, db = _dom(la), _dom(lb)
        if da is not None and da == db:            # reachable real split on a held-out neuron
            if _pair_key(la, lb) in accepted: fixed += 1
            else: remaining += 1

    return {
        "brain": brain_id, "image_on": image_on,
        "n_heldout": len(heldout_names), "n_train": len(train_names),
        "n_split_sites": len(split_sites), "n_edits": len(edits),
        "base": base, "edited": edited, "repair": repair,
        "heldout_names": heldout_names,
        "coverage": {"fixed": fixed, "remaining": remaining,
                     "reachable": fixed + remaining},
        "split_sites": split_sites, "edits": edits, "accepted": accepted,
        "heldout_map": heldout_map,
    }

## 3. Run every test brain

In [4]:
results = {}
for b in TEST_BRAINS:
    print(f"\n=== analyzing test brain {b} ===")
    results[b] = analyze_brain(b)
    r = results[b]
    img = "image ON" if r["image_on"] else "GEOMETRY-ONLY (image off)"
    print(f"  {b}: {r['n_heldout']} held-out neurons, {r['n_split_sites']} candidate pairs, "
          f"{r['n_edits']} merge edits  [{img}]")


=== analyzing test brain 794491 ===
  794491: 9 held-out neurons, 25000 candidate pairs, 8656 merge edits  [image ON]


## 4. Per-skeleton metric comparison (baseline vs edited) — per test brain

Every row is a held-out neuron of the test brain. `d…` columns are edited − baseline;
`edit-caused merges` is how many cross-neuron fusions the policy INTRODUCED on that
neuron (the cause behind `d% Merged Edges`, which can net to ~0 when new merges
cancel pre-existing ones).

In [5]:
COLS = ["Edge Accuracy", "% Merged Edges", "# Merges", "% Split Edges", "# Splits", "% Omit Edges"]

def _edit_caused(edited, index):
    ms = getattr(edited, "merge_sites", None)
    if ms is None or len(ms) == 0 or "GroundTruth_ID" not in getattr(ms, "columns", []) \
            or "Caused_By_Edit" not in ms.columns:
        return pd.Series(0, index=index, dtype=int)
    rows = ms[ms["Caused_By_Edit"]]
    return rows["GroundTruth_ID"].value_counts().reindex(index).fillna(0).astype(int)

def comparison_table(r):
    base, edited = r["base"], r["edited"]
    b = base.per_swc[COLS]
    e = edited.per_swc.reindex(b.index)[COLS]
    delta = (e - b).add_prefix("d")
    comp = pd.concat([b.add_prefix("base "), e.add_prefix("edit "), delta], axis=1)
    comp["edit-caused merges"] = _edit_caused(edited, b.index)
    comp.loc["WEIGHTED MEAN"] = {
        **{f"base {c}": scoring._weighted_avg(base.per_swc, c) for c in COLS},
        **{f"edit {c}": scoring._weighted_avg(edited.per_swc, c) for c in COLS},
        **{f"d{c}": scoring._weighted_avg(edited.per_swc, c) - scoring._weighted_avg(base.per_swc, c) for c in COLS},
    }
    comp.loc["WEIGHTED MEAN", "edit-caused merges"] = int(comp["edit-caused merges"][:-1].sum())
    return comp

pd.set_option("display.width", 220, "display.max_columns", 40)
for b in TEST_BRAINS:
    print(f"\n################  TEST BRAIN {b}  ################")
    display(comparison_table(results[b]).round(3))


################  TEST BRAIN 794491  ################


,base Edge Accuracy,base % Merged Edges,base # Merges,base % Split Edges,base # Splits,base % Omit Edges,edit Edge Accuracy,edit % Merged Edges,edit # Merges,edit % Split Edges,edit # Splits,edit % Omit Edges,dEdge Accuracy,d% Merged Edges,d# Merges,d% Split Edges,d# Splits,d% Omit Edges,edit-caused merges
N001-794491-JG,68.940,29.949,13.000,0.423,801.000,0.689,68.890,30.000,13.000,0.422,798.00,0.689,-0.050,0.052,0.000,-0.001,-3.000,0.000,5.0
N002-794491-PP,72.550,25.029,29.000,0.677,1070.000,1.739,72.350,25.242,30.000,0.670,1058.00,1.737,-0.200,0.213,1.000,-0.007,-12.000,-0.003,10.0
N004-794491-MB,69.380,27.684,21.000,0.563,628.000,2.373,63.080,33.989,23.000,0.558,622.00,2.370,-6.300,6.305,2.000,-0.005,-6.000,-0.004,10.0
N005-794491-HP,46.660,52.368,3.000,0.330,398.000,0.637,46.670,52.368,3.000,0.327,397.00,0.629,0.010,0.000,0.000,-0.003,-1.000,-0.008,0.0
N006-794491-SP,96.130,3.341,4.000,0.313,471.000,0.219,96.140,3.341,4.000,0.310,468.00,0.214,0.010,0.000,0.000,-0.003,-3.000,-0.004,1.0
N008-794491-SP,68.490,27.630,37.000,0.911,1617.000,2.965,67.130,28.995,37.000,0.906,1608.00,2.965,-1.360,1.365,0.000,-0.006,-9.000,0.000,17.0
N009-794491-JG,80.880,10.130,9.000,1.468,1541.000,7.526,79.960,11.048,10.000,1.466,1536.00,7.526,-0.920,0.918,1.000,-0.002,-5.000,0.000,1.0
N012-794491-SP,82.230,14.536,17.000,0.868,1738.000,2.364,81.920,14.865,18.000,0.862,1725.00,2.355,-0.310,0.329,1.000,-0.006,-13.000,-0.009,2.0
N013-794491-PP,81.310,7.488,15.000,1.755,3104.000,9.443,81.260,7.564,15.000,1.745,3083.00,9.427,-0.050,0.076,0.000,-0.009,-21.000,-0.017,1.0
WEIGHTED MEAN,75.014,21.102,17.332,0.824,1354.763,3.059,74.217,21.910,17.835,0.819,1345.82,3.054,-0.797,0.808,0.503,-0.005,-8.944,-0.005,47.0


## 5. Split-repair coverage on the test brain (the headline generalization number)

In [6]:
def _pct(n, d): return f"{100.0*n/d:.1f}%" if d else "--"

summary_rows = []
for b in TEST_BRAINS:
    r = results[b]; rep = r["repair"]; cov = r["coverage"]
    img = "" if r["image_on"] else "   [GEOMETRY-ONLY — image off, NOT the true score]"
    print(f"\n=== TEST BRAIN {b} (whole-brain held-out){img} ===")
    print(f"  PER-EDIT split-repair (gate signal): correct={rep['correct']}  "
          f"false={rep['false']}  unscored={rep['unscored']}  "
          f"score(correct-false)={rep['correct']-rep['false']}")
    print(f"  PER-SITE coverage: {cov['fixed']} of {cov['reachable']} distinct reachable "
          f"split errors fixed ({_pct(cov['fixed'], cov['reachable'])}); "
          f"{cov['remaining']} reachable-but-unrepaired")
    base_acc = scoring._weighted_avg(r["base"].per_swc, "Edge Accuracy")
    edit_acc = scoring._weighted_avg(r["edited"].per_swc, "Edge Accuracy")
    base_mrg = scoring._weighted_avg(r["base"].per_swc, "% Merged Edges")
    edit_mrg = scoring._weighted_avg(r["edited"].per_swc, "% Merged Edges")
    print(f"  Edge Accuracy {base_acc:.3f} -> {edit_acc:.3f} ({edit_acc-base_acc:+.3f}); "
          f"% Merged Edges {base_mrg:.3f} -> {edit_mrg:.3f} ({edit_mrg-base_mrg:+.3f})")
    summary_rows.append({
        "test_brain": b, "image_on": r["image_on"], "n_heldout": r["n_heldout"],
        "correct": rep["correct"], "false": rep["false"], "unscored": rep["unscored"],
        "split_repair_score": rep["correct"] - rep["false"],
        "reachable": cov["reachable"], "fixed": cov["fixed"], "remaining": cov["remaining"],
        "coverage_pct": (100.0*cov["fixed"]/cov["reachable"] if cov["reachable"] else float("nan")),
        "base_edge_acc": base_acc, "edit_edge_acc": edit_acc,
        "base_pct_merged": base_mrg, "edit_pct_merged": edit_mrg,
    })

summary = pd.DataFrame(summary_rows).set_index("test_brain")
print("\n=== cross-brain test summary ===")
display(summary.round(3))


=== TEST BRAIN 794491 (whole-brain held-out) ===
  PER-EDIT split-repair (gate signal): correct=73  false=0  unscored=8583  score(correct-false)=73
  PER-SITE coverage: 73 of 203 distinct reachable split errors fixed (36.0%); 130 reachable-but-unrepaired
  Edge Accuracy 75.014 -> 74.217 (-0.797); % Merged Edges 21.102 -> 21.910 (+0.808)

=== cross-brain test summary ===


,image_on,n_heldout,correct,false,unscored,split_repair_score,reachable,fixed,remaining,coverage_pct,base_edge_acc,edit_edge_acc,base_pct_merged,edit_pct_merged
test_brain,,,,,,,,,,,,,,
794491,True,9,73,0,8583,73,203,73,130,35.961,75.014,74.217,21.102,21.91


## 6. Save the test-brain analysis to the run folder

In [7]:
ana_dir = RUN_DIR / "test_brain_analysis"
ana_dir.mkdir(parents=True, exist_ok=True)

# Tidy summary CSV across test brains.
summary_csv = ana_dir / f"test_brain_summary_{GEN}.csv"
summary.to_csv(summary_csv)

# Per-brain JSON (coverage + per-edit + metric deltas) and per-skeleton CSV.
for b in TEST_BRAINS:
    r = results[b]; rep = r["repair"]; cov = r["coverage"]
    payload = {
        "run": RUN_NAME, "generation": GEN, "test_brain": b,
        "cross_brain": CROSS_BRAIN, "splits_only": SPLITS_ONLY, "mcl": MCL,
        "image_reader_enabled": r["image_on"],
        "n_heldout_neurons": r["n_heldout"], "n_distinct_candidate_pairs": r["n_split_sites"],
        "n_merge_labels_edits": r["n_edits"],
        "per_edit": {k: rep[k] for k in ("correct", "false", "unscored")},
        "per_site": cov,
    }
    (ana_dir / f"test_brain_coverage_{GEN}_{b}.json").write_text(json.dumps(payload, indent=2))
    comparison_table(r).to_csv(ana_dir / f"test_brain_per_skeleton_{GEN}_{b}.csv")

print(f"saved summary -> {summary_csv}")
print(f"saved per-brain JSON + per-skeleton CSV under -> {ana_dir}")
if not all(results[b]["image_on"] for b in TEST_BRAINS):
    print("[WARN] one or more brains scored with image OFF — those rows are the "
          "geometry-only subset, not the policy's true behaviour. Re-run with GCS access.")

saved summary -> /allen/programs/mindscope/workgroups/auto-model/zihan.zhang/exaspim-agent/exa-spim-agent/proofreader_evolve/runs/789202_train1_test1_20260628_212929/test_brain_analysis/test_brain_summary_gen39.csv
saved per-brain JSON + per-skeleton CSV under -> /allen/programs/mindscope/workgroups/auto-model/zihan.zhang/exaspim-agent/exa-spim-agent/proofreader_evolve/runs/789202_train1_test1_20260628_212929/test_brain_analysis
